# 1) MLP: A * B % C (A,B between 0..99, C between 0..19)

In [12]:
#@title (0) Setup
from google.colab import drive
import os

if not os.path.exists('/content/drive/MyDrive'):
  drive.mount('/content/drive')

%run "/content/drive/MyDrive/Colab Notebooks/Build your own LLM/han-common.ipynb"

Train: 152,000 samples  |  Test: 38,000 samples
Input dim: 219  |  Output classes: 19
Label distribution (train): {np.int64(0): np.int64(51411), np.int64(1): np.int64(11977), np.int64(2): np.int64(14363), np.int64(3): np.int64(10413), np.int64(4): np.int64(11615), np.int64(5): np.int64(6717), np.int64(6): np.int64(10053), np.int64(7): np.int64(4823), np.int64(8): np.int64(6869), np.int64(9): np.int64(4650), np.int64(10): np.int64(4769), np.int64(11): np.int64(2542), np.int64(12): np.int64(4452), np.int64(13): np.int64(1747), np.int64(14): np.int64(2023), np.int64(15): np.int64(1374), np.int64(16): np.int64(1254), np.int64(17): np.int64(538), np.int64(18): np.int64(410)}
DNN(
  (proj_in): Linear(in_features=219, out_features=128, bias=True)
  (layers): ModuleList(
    (0-15): 16 x Sequential(
      (0): Linear(in_features=128, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=128, bias=True)
    )
  )
  (norms): ModuleList(
    (0-15): 16 x L

## 1.2) DNN: 219-> 128-> 64 -> 32 -> 19


In [13]:
class DNN(nn.Module):
    def __init__(self):
        super().__init__()
        hidden_dim = 128
        self.proj_in = nn.Linear(INPUT_DIM, hidden_dim)
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        for _ in range(1):
            self.layers.append(
                nn.Sequential(
                    nn.Linear(hidden_dim, hidden_dim // 2),
                    nn.ReLU(),
                    nn.Linear(hidden_dim // 2, hidden_dim // 4))
                )
            self.norms.append(nn.RMSNorm(hidden_dim))
        self.proj_out = nn.Linear(hidden_dim // 4, NUM_CLASSES)

    def forward(self, x):
        x = F.relu(self.proj_in(x))
        for layer, norm in zip(self.layers, self.norms):
            x = layer(norm(x))  # Removed residual addition due to dimension change
        return self.proj_out(x)

MODEL = DNN().to(DEVICE)
print(MODEL)

print(f"\nTotal parameters: {sum(p.numel() for p in MODEL.parameters()):,}")
print(f"Per-layer shapes:")
for name, param in MODEL.named_parameters():
    if "weight" in name:
        print(f"  {name:30s}  {str(list(param.shape)):>15s}  ({param.numel():>6,} params)")


DNN(
  (proj_in): Linear(in_features=219, out_features=128, bias=True)
  (layers): ModuleList(
    (0): Sequential(
      (0): Linear(in_features=128, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=32, bias=True)
    )
  )
  (norms): ModuleList(
    (0): RMSNorm((128,), eps=None, elementwise_affine=True)
  )
  (dropouts): ModuleList()
  (proj_out): Linear(in_features=32, out_features=19, bias=True)
)

Total parameters: 39,251
Per-layer shapes:
  proj_in.weight                       [128, 219]  (28,032 params)
  layers.0.0.weight                     [64, 128]  ( 8,192 params)
  layers.0.2.weight                      [32, 64]  ( 2,048 params)
  norms.0.weight                            [128]  (   128 params)
  proj_out.weight                        [19, 32]  (   608 params)


In [14]:
#@title visualize architecture

import graphviz
from IPython.display import display

def visualize_architecture_enhanced():
    # Use SVG for sharper rendering and ortho lines for cleaner routing
    dot = graphviz.Digraph(comment='DNN Architecture', format='svg')
    # Adjusted size and ranksep to reduce overall height
    dot.attr(rankdir='TB', size='10,8.5', bgcolor='#0B1120', fontname='Helvetica', splines='ortho', nodesep='0.8', ranksep='0.4')

    # Global node and edge style
    dot.attr('node', shape='plaintext', fontname='Helvetica')
    dot.attr('edge', color='#64748B', arrowsize='0.8', penwidth='2')

    # Helper function to create beautiful HTML-like nodes
    def html_node(name, title, details, color_top, color_bottom):
        label = f'''<<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0" CELLPADDING="8">
        <TR><TD BGCOLOR="{color_top}" WIDTH="220"><FONT COLOR="white" POINT-SIZE="14"><B> {title} </B></FONT></TD></TR>
        <TR><TD BGCOLOR="{color_bottom}" WIDTH="220"><FONT COLOR="#E2E8F0" POINT-SIZE="12"> {details} </FONT></TD></TR>
        </TABLE>>'''
        dot.node(name, label)

    # Define nodes with rich colors
    html_node('Input', 'INPUT FEATURES', 'Dimension: 219', '#1E293B', '#334155')
    html_node('ProjIn', 'PROJECTION IN', 'Linear(219 → 128)<BR/>+ ReLU', '#2563EB', '#1D4ED8')

    # Pre-Norm FFN Block
    with dot.subgraph(name='cluster_block') as c:
        # Reduced margin to compress height further
        c.attr(style='rounded,dashed', color='#475569', fontcolor='#F8FAFC',
               label='  Pre-Norm FFN Block (x1)  ', margin='15', penwidth='2', fontsize='14')

        c.node('Split', '', shape='point', width='0')

        # Internal block nodes
        c.node('Norm', '''<<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0" CELLPADDING="8">
        <TR><TD BGCOLOR="#7C3AED" WIDTH="200"><FONT COLOR="white"><B> RMSNorm </B></FONT></TD></TR>
        <TR><TD BGCOLOR="#6D28D9" WIDTH="200"><FONT COLOR="#E2E8F0"> Features: 128 </FONT></TD></TR>
        </TABLE>>''')

        c.node('FFN1', '''<<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0" CELLPADDING="8">
        <TR><TD BGCOLOR="#059669" WIDTH="200"><FONT COLOR="white"><B> FFN Layer 1 </B></FONT></TD></TR>
        <TR><TD BGCOLOR="#047857" WIDTH="200"><FONT COLOR="#E2E8F0"> Linear(128 → 256)<BR/>+ ReLU </FONT></TD></TR>
        </TABLE>>''')

        c.node('FFN2', '''<<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0" CELLPADDING="8">
        <TR><TD BGCOLOR="#059669" WIDTH="200"><FONT COLOR="white"><B> FFN Layer 2 </B></FONT></TD></TR>
        <TR><TD BGCOLOR="#047857" WIDTH="200"><FONT COLOR="#E2E8F0"> Linear(256 → 128) </FONT></TD></TR>
        </TABLE>>''')

        # Beautiful Add node
        c.node('Add', '⊕', shape='circle', style='filled', fillcolor='#DB2777',
               color='white', fontcolor='white', penwidth='2', width='0.6', fontsize='24')

        # Internal connections
        c.edge('Split', 'Norm')
        c.edge('Norm', 'FFN1')
        c.edge('FFN1', 'FFN2')
        c.edge('FFN2', 'Add')
        # Residual skip connection - using xlabel to fix warning
        c.edge('Split', 'Add', style='dashed', xlabel=' Residual Skip ',
               fontcolor='#94A3B8', color='#94A3B8', fontsize='12')

    html_node('ProjOut', 'PROJECTION OUT', 'Linear(128 → 19)', '#EA580C', '#C2410C')
    html_node('Output', 'OUTPUT LOGITS', 'Dimension: 19', '#BE123C', '#9F1239')

    # Main flow connections
    dot.edge('Input', 'ProjIn')
    dot.edge('ProjIn', 'Split', dir='none', penwidth='2')
    dot.edge('Add', 'ProjOut', penwidth='2')
    dot.edge('ProjOut', 'Output')

    return dot

# Display the enhanced architecture graph
display(visualize_architecture_enhanced())


## 1.3) Train

In [15]:
def train(run_name=""):
    model = DNN().to(DEVICE)
    model = train_common(model, run_name)

    history = get_history(run_name)
    if history is not None:
        visualize_individual_run(history, run_name)
    else:
        print(f"Warning: Could not find history for run '{run_name}' in all_runs.")

    return model

# 2) Train and smoke test

In [16]:
MODEL = train()

  🚀 Run: 
Epoch      1 | Train loss 1.0588  acc 55.0% | Test loss 1.0712  acc 54.0%
Epoch      2 | Train loss 0.9301  acc 62.1% | Test loss 0.9441  acc 61.4%
Epoch      3 | Train loss 0.8501  acc 65.3% | Test loss 0.8592  acc 64.6%
Epoch      4 | Train loss 0.7449  acc 71.0% | Test loss 0.7517  acc 70.6%
Epoch      5 | Train loss 0.6095  acc 76.1% | Test loss 0.6143  acc 75.8%
Epoch      6 | Train loss 0.5725  acc 78.2% | Test loss 0.5774  acc 77.8%
Epoch      7 | Train loss 0.5121  acc 82.1% | Test loss 0.5165  acc 81.8%
Epoch      8 | Train loss 0.4227  acc 85.2% | Test loss 0.4272  acc 84.9%
Epoch      9 | Train loss 0.3543  acc 88.0% | Test loss 0.3563  acc 87.8%
Epoch     10 | Train loss 0.3109  acc 89.6% | Test loss 0.3136  acc 89.4%
Epoch     11 | Train loss 0.3111  acc 90.2% | Test loss 0.3121  acc 90.0%
Epoch     12 | Train loss 0.2575  acc 91.2% | Test loss 0.2593  acc 91.2%
Epoch     13 | Train loss 0.2332  acc 91.9% | Test loss 0.2361  acc 91.8%
Epoch     14 | Train loss 0.

In [17]:
#@title find interesting A, B, C
import scipy.stats

MODEL.eval()
best_diff = float('inf')
best_abc = None
best_logits = None
best_probs = None

# Sample some combinations to find one where the top prediction is around 50%
with torch.no_grad():
    for i in range(1000):
        a, b, c = random.randint(0, 99), random.randint(0, 99), random.randint(0, 18)
        x = np.concatenate([one_hot(a), one_hot(b), one_hot(c, 19)])
        x_tensor = torch.tensor(x).unsqueeze(0).to(DEVICE)

        logits = MODEL(x_tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

        # Find the max probability
        max_prob = np.max(probs)

        # We want the max probability to be around 50% (0.5)
        diff = abs(max_prob - 0.5)

        if diff < best_diff:
            best_diff = diff
            best_abc = (a, b, c)
            best_logits = logits
            best_probs = probs

print(f"Found interesting combination: A={best_abc[0]}, B={best_abc[1]}, C={best_abc[2]}")
print(f"Expected result: {(best_abc[0] * best_abc[1]) % (best_abc[2] + 1)}")
print(f"Max Probability: {np.max(best_probs)*100:.1f}%")

# We can plot these probabilities using the existing visualization code if we set `logits = best_logits`
logits = best_logits

Found interesting combination: A=27, B=19, C=15
Expected result: 1
Max Probability: 59.1%


# 3) get logits from model

In [18]:
MODEL.eval()
with torch.no_grad():
    a_add_b_mod_c = torch.tensor(np.concatenate([
        one_hot(72), one_hot(12), one_hot(10, 19)])
    ).unsqueeze(0).to(DEVICE)

    logits = MODEL(a_add_b_mod_c)

    print(f"Prediction: {logits} (Expected: {(72 * 12) % (10 + 1)})")

Prediction: tensor([[-2.0797,  0.5498,  3.3711,  2.8652, -4.7040, -2.2644,  9.6529,  0.1134,
          3.6193, -7.0543,  0.6379, -2.6196,  0.0317, -5.5710, -4.5117, -3.7426,
         -1.7828, -6.0857, -6.7608]], device='cuda:0') (Expected: 6)


# 4) SoftMax (pyTorch)


In [19]:
probs = torch.softmax(logits, dim=1)

print("Softmax Probabilities:")
formatted_probs = [f"{p.item() * 100:.1f}%" for p in probs[0]]
print(formatted_probs)

Softmax Probabilities:
['0.0%', '0.0%', '0.2%', '0.1%', '0.0%', '0.0%', '99.4%', '0.0%', '0.2%', '0.0%', '0.0%', '0.0%', '0.0%', '0.0%', '0.0%', '0.0%', '0.0%', '0.0%', '0.0%']


In [ ]:
#@title visualize
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.colors as mcolors
import logging

# Suppress font missing warnings
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR);

# Prefer the deployment font when available, otherwise fall back to common sans-serif fonts
font_candidates = ['Inter', 'DejaVu Sans', 'Arial', 'Helvetica', 'sans-serif']
plt.rcParams['font.family'] = font_candidates
font_path = 'fonts/Inter-Regular.ttf'
try:
    prop = fm.FontProperties(fname=font_path)
    plt.rcParams['font.family'] = prop.get_name()
except Exception:
    pass

# Set styling for all text and lines
plt.rcParams['text.color'] = '#F1F5F9'
plt.rcParams['axes.labelcolor'] = '#F1F5F9'
plt.rcParams['xtick.color'] = '#F1F5F9'
plt.rcParams['ytick.color'] = '#F1F5F9'
plt.rcParams['axes.edgecolor'] = '#F1F5F9'

# Grid styling
plt.rcParams['grid.color'] = '#5D6674'
plt.rcParams['grid.alpha'] = 0.7

# ======== DATA ========
# Extract probabilities from the previous tensor and convert to percentages
probs_np = probs[0].cpu().numpy() * 100
categories = [str(i) for i in range(len(probs_np))]  # Actual C values (0 to 18)
values_softmax = probs_np.tolist()

# Calculate linear probabilities from logits
logits_np = logits[0].cpu().numpy()
logits_min = np.min(logits_np)
linear_probs_np = (logits_np - logits_min) / np.sum(logits_np - logits_min) * 100
values_linear = linear_probs_np.tolist()
# ======================

# Create figure
fig, ax = plt.subplots(figsize=(16, 6), dpi=300)
fig.patch.set_facecolor('#1E293B')
ax.set_facecolor('#1E293B')

# Bar configuration
x = np.arange(len(categories))
width = 0.4

color_softmax = '#8C7BEF'
color_linear = '#F87171'

bars1 = ax.bar(x - width/2, values_softmax, width, label='Softmax Probability', color=color_softmax, edgecolor='#F1F5F9', linewidth=1.0)
bars2 = ax.bar(x + width/2, values_linear, width, label='Linear Probability', color=color_linear, edgecolor='#F1F5F9', linewidth=1.0)

ax.set_title('Softmax vs Linear Probabilities', fontsize=16, color='#F1F5F9', pad=15)
ax.set_xlabel('C Value', fontsize=15, labelpad=10, color='#F1F5F9')
ax.set_ylabel('Probability (%)', fontsize=15, color='#F1F5F9', labelpad=10)
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.grid(axis='y', alpha=0.7, color='#5D6674', linestyle='--', linewidth=0.5, zorder=0)
ax.set_axisbelow(True)

# Add text labels on top of the bars, rotated so they don't overlap
def add_labels(bars):
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
                 f'{height:.1f}%', ha='center', va='bottom', fontsize=9, color='#F1F5F9', rotation=90)

add_labels(bars1)
add_labels(bars2)

# Adjust y-limit to make room for rotated text
max_val = max(max(values_softmax), max(values_linear))
ax.set_ylim(0, max_val + max(5, max_val * 0.25))

# Add Legend
ax.legend(facecolor='#1E293B', edgecolor='#5D6674', labelcolor='#F1F5F9', fontsize=12)

# Spines styling
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)
ax.spines['bottom'].set_color('#F1F5F9')
ax.spines['left'].set_color('#F1F5F9')
ax.spines['left'].set_position(('outward', 2))
ax.spines['bottom'].set_position(('outward', 2))

plt.tight_layout()
plt.show()

# 5) Exercise: implement softmax function

In [ ]:
sample_logits = torch.tensor([[ -5.4170,  -1.9652,   3.3076,  -2.9467,  -0.5578,  -1.4257,   3.3773,
           2.7374,   3.7214,  -2.4323,   2.6508, -12.6265,  -3.4000,  -6.8072,
          -5.4235, -10.6900,  -7.3629, -10.2123,  -9.2929]])

assert False, "implement mySoftmax"

print(mySoftmax(logits))

AssertionError: implement mySoftmax

In [ ]:
def mySoftmax(logits):
    if not isinstance(logits, torch.Tensor):
        logits = torch.tensor(logits)

    max_logits = torch.max(logits, dim=1, keepdim=True).values
    exp_logits = torch.exp(logits - max_logits)
    sum_exp_logits = torch.sum(exp_logits, dim=1, keepdim=True)
    probs = exp_logits / sum_exp_logits

    return probs

probs = mySoftmax(sample_logits)
print([f"{p.item() * 100:.1f}%" for p in probs[0]])

['0.0%', '0.1%', '21.2%', '0.0%', '0.4%', '0.2%', '22.8%', '12.0%', '32.1%', '0.1%', '11.0%', '0.0%', '0.0%', '0.0%', '0.0%', '0.0%', '0.0%', '0.0%', '0.0%']


In [20]:
#@title (6) Save and load model
FILENAME = "DDN-v8.pth"

print("--- Testing Local Save & Load ---")
load_model = DNN().to(DEVICE)
load_model = save_load_model_local(MODEL, load_model, filename_c=FILENAME)
predicted_value = invoke_model_with_a_b_c(load_model, 18, 1, 10)

--- Testing Local Save & Load ---
model weights saved locally to: .//DDN-v8.pth
model weights loaded successfully into a new model instance.
Prediction for (18 * 1) % 10 = 8


In [21]:
#@title (7) Save to Google Drive
pathname = save_to_google_drive(MODEL, FILENAME)

Model saved to Google Drive: /content/drive/MyDrive/workshop/DDN-v8.pth


In [ ]:
#@title (8) Load Pretrained model from Hugging Face
print("\n--- Testing Hugging Face Download & Load ---")
hf_model = DNN().to(DEVICE)
hf_model = load_model_hugging_face(new_model_c=hf_model, filename_c=FILENAME)
predicted_value = invoke_model_with_a_b_c(hf_model, 22, 1, 10)

MODEL = hf_model
